<a href="https://colab.research.google.com/github/Urooj25/Movie-review-sentiment-analysis/blob/main/Copy_of_Movie_Review_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# --- 1. Load & Preprocess Data ---
df = pd.read_csv("/bin/IMDB Dataset.csv")
# Mapping: Negative -> -1, Neutral -> 0, Positive -> 1
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': -1, 'neutral': 0})

# Setup NLTK
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = re.sub(r'<.*?>', '', text) # Remove HTML
    text = re.sub(r'[^a-zA-Z]', ' ', text).lower() # Clean & Lowercase
    words = text.split()
    return " ".join([lemmatizer.lemmatize(w) for w in words if w not in stop_words])

df['cleaned_review'] = df['cleaned_review'] = df['review'].apply(clean_text)

# --- 2. Feature Engineering (TF-IDF + SVD) ---
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['cleaned_review'])

# SVD for dimensionality reduction
svd = TruncatedSVD(n_components=200, random_state=42)
X_svd = svd.fit_transform(X)
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(X_svd, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(
    n_estimators=300,           # More trees = more stability
    max_depth=8,                # Very shallow to prevent memorization
    min_samples_leaf=20,        # Each leaf needs at least 20 samples
    min_samples_split=50,       # Don't even try to split if samples are low
    max_features='log2',        # Limit the number of features per tree
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

rf_model = RandomForestClassifier(
    n_estimators=500,           # High number of trees to smooth out the noise
    max_depth=5,                # Very shallow: forces the model to be simple
    min_samples_leaf=50,        # A rule must apply to at least 50 reviews
    max_features=10,            # Only look at 10 features at a time (very restrictive)
    ccp_alpha=0.001,            # Prunes the trees to keep them small
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'     # Helps if neutral/positive/negative classes are uneven
)

rf_model.fit(X_train, y_train)

# --- Check New Results ---
train_acc = accuracy_score(y_train, rf_model.predict(X_train))
test_acc = accuracy_score(y_test, rf_model.predict(X_test))

print(f"Final Training Accuracy: {train_acc * 100:.2f}%")
print(f"Final Testing Accuracy:  {test_acc * 100:.2f}%")
print(f"Gap: {(train_acc - test_acc) * 100:.2f}%")

# --- 4. Evaluation ---
train_acc = accuracy_score(y_train, rf_model.predict(X_train))
test_acc = accuracy_score(y_test, rf_model.predict(X_test))

print(f"Training Accuracy: {train_acc * 100:.2f}%")
print(f"Testing Accuracy:  {test_acc * 100:.2f}%")

if train_acc > test_acc + 0.05:
    print("Status: Slight Overfitting - Consider lowering max_depth.")
else:
    print("Status: Model is well-generalized.")

# --- 5. Recommendation Logic ---
def get_recommendation(review_text):
    clean_input = clean_text(review_text)
    vec_input = svd.transform(tfidf.transform([clean_input]))

    # Random Forest gives probabilities for each class [-1, 0, 1]
    probs = rf_model.predict_proba(vec_input)[0]
    prediction = rf_model.predict(vec_input)[0]

    actions = {-1: "Do not recommend", 0: "Maybe watch (Neutral)", 1: "Recommend movie"}
    return f"Action: {actions[prediction]} (Confidence: {max(probs):.2f})"

# Test
user_review = input("Enter your review: ")
print(get_recommendation(user_review))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Final Training Accuracy: 78.44%
Final Testing Accuracy:  77.44%
Gap: 1.01%
Training Accuracy: 78.44%
Testing Accuracy:  77.44%
Status: Model is well-generalized.
Enter your review: bad movie
Action: Do not recommend (Confidence: 0.63)


### Implementing Decision Tree from Scratch

First, let's implement the core components of a Decision Tree, including entropy calculation, information gain, and the tree building process.

### Testing Decision Tree from Scratch

Now, let's train our custom Decision Tree model and evaluate its accuracy.

### Implementing Random Forest from Scratch

Next, we'll build a Random Forest by combining multiple Decision Trees using bootstrap aggregation (bagging).

This profiling cell will run the `RandomForest.fit` method and then display the top 10 functions that consumed the most cumulative time. Look at the `cumtime` column to identify bottlenecks. You can adjust the `n_trees`, `max_depth`, and `n_features` in `rf_model_profile` to run a quicker profile or a full one, depending on your needs.

### Testing Random Forest from Scratch

Finally, let's train our custom Random Forest model and evaluate its accuracy.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')